# 02 · Preprocessing research and matched exports

**Inputs:** a frozen protocol, source training/calibration images and phase-01
converted development labels/Raw references. **Outputs:** nested subsets,
paired S10 datasets, frozen policy, paired development inputs, quality/latency plots
and compact evidence under `runs/preprocessing-v1/` / `docs/preprocessing-v1/`.

Set the artifact paths below, then **Restart & Run All**. No data initialization,
training or detector inference is invoked. Missing phase-01 artifacts fail early.
The existing evidence-selected **v2 Gamma-or-Skip** policy is inspected and reused;
changing it requires a separate approved calibration experiment. Calibration-only
operator ablations and synthetic evidence remain inspectable in the archived research
snapshots. Development images are exported with the frozen policy, never used to fit it.

In [ ]:
import gzip
import os
import platform
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

from vehicle_detection.adaptive import (
    METRICS,
    export_pair,
    paired_quality,
    validate_policy,
)
from vehicle_detection.common import (
    file_hash,
    freeze_json,
    read_image,
    read_json,
    require_paths,
    write_json,
)
from vehicle_detection.data import CLASSES, convert_labels, scan_bdd
from vehicle_detection.datasets import export_development_pair
from vehicle_detection.offline import estimate_disk, verify_alignment
from vehicle_detection.protocol import validate_protocol
from vehicle_detection.subsets import build_subsets, freeze_subsets, training_records

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
DATA_ROOT = ROOT / "data/bdd100k"
RUN_ROOT = ROOT / "runs/preprocessing-v1"
REPORT_ROOT = ROOT / "docs/preprocessing-v1"
PROTOCOL_PATH = ROOT / "runs/data-v1/protocol.json"
SEED, ATTEMPTS = 2026, 128
INCLUDE_S40 = False
EXPORT_SUBSET = "S10"  # S20 is a manifest only until a later scaling decision.
CALIBRATION_SAMPLE, PER_OPERATOR, MIN_SUPPORT = 256, 12, 12
FROZEN_POLICY = ROOT / "docs/issue6-v2/policy.json"
RAW_DEVELOPMENT = ROOT / "runs/data-v1/datasets/raw"
DEV_CONVERSION = ROOT / "runs/data-v1/conversion"
require_paths(
    protocol=PROTOCOL_PATH,
    data=DATA_ROOT,
    policy=FROZEN_POLICY,
    raw_development=RAW_DEVELOPMENT / "verified.json",
    development_labels=DEV_CONVERSION / "conversion.json",
    decision=ROOT / "docs/issue11-v2/v3_decision.json",
    screening=ROOT / "docs/issue11-v2/experimental_matrix.csv",
    screening_figure=ROOT / "docs/issue11-v2/supported_recovery.png",
)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
cv2.setNumThreads(1)
plt.rcParams.update({"figure.dpi": 90, "font.size": 9})
protocol = validate_protocol(read_json(PROTOCOL_PATH))
print("Frozen parent:", protocol["manifest_id"])
print("No detector, training, development or final-test image access in this notebook.")

## Freeze nested, representative image-ID subsets

Search 128 seeded whole-group orders for minimum combined S10/S20 drift in class image
presence and instances, negatives, object sizes, weather, time, scene, their intersections
and class × condition support. Prefixes enforce nesting; whole groups can move nominal
image percentages. The full training pool is the only selection reference. Rare/zero
strata are disclosed, never manufactured by oversampling. Inferred sequence grouping
and duplicate safeguards inherit the limitations of issue #5.

In [ ]:
subsets = build_subsets(protocol, seed=SEED, attempts=ATTEMPTS, include_s40=INCLUDE_S40)
freeze_subsets(RUN_ROOT / "subsets.json", subsets, protocol)
pool = protocol["splits"]["training_pool"]
counts = pd.DataFrame(
    {
        "training_pool": protocol["summaries"]["training_pool"],
        **{k: v["summary"] for k, v in subsets["subsets"].items()},
    }
).fillna(0)
display(counts.loc[["images", "negative", *[f"presence:{c}" for c in CLASSES]]].T)
print("Subset identity:", subsets["manifest_id"])
display(
    pd.DataFrame(
        [
            {
                "subset": k,
                "groups": len(v["groups"]),
                "target_fraction": v["target_fraction"],
                "actual_fraction": v["actual_fraction"],
            }
            for k, v in subsets["subsets"].items()
        ]
    )
)
audit = pd.DataFrame(subsets["diagnostics"])
audit.to_csv(REPORT_ROOT / "subset_support.csv", index=False)
display(audit[audit.rare_support].sort_values(["feature", "subset"]))
display(audit.assign(abs_drift=audit.delta_per_image.abs()).nlargest(15, "abs_drift"))
print("\n".join(subsets["limitations"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, prefix in zip(axes, ["presence", "instances"], strict=True):
    data = counts.reindex([f"{prefix}:{c}" for c in CLASSES], fill_value=0)
    data.index = CLASSES
    data.div(counts.loc["images"]).plot.bar(ax=ax, title=f"Class {prefix} / image", rot=20)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "class_support.png")
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, axis in zip(axes, ["weather", "timeofday", "scene"], strict=True):
    data = counts.loc[[k for k in counts.index if k.startswith(axis + ":")]]
    data.index = data.index.str.split(":", n=1).str[1]
    data.div(counts.loc["images"]).plot.bar(ax=ax, title=axis, rot=65)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "conditions.png")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
counts.reindex([f"size_instances:{s}" for s in ["small", "medium", "large"]], fill_value=0).div(
    counts.loc["images"]
).plot.bar(ax=axes[0], title="Box-size histogram / image", rot=20)
counts.loc[["negative"]].div(counts.loc["images"]).T.plot.bar(
    ax=axes[1], title="Negative image fraction", legend=False, rot=20
)
fig.tight_layout()
fig.savefig(REPORT_ROOT / "sizes_negatives.png")
plt.show()

## Inspect the frozen calibration and independent operator ablations

The following measurements/candidates are the original calibration evidence,
not new scientific results. V2 compared each mild operator against matched JPEG95
skip on calibration-only IDs with a minimum-support gate. Gamma-or-Skip won;
unsupported CLAHE/Denoise/Sharpen routes remain disabled. The noise proxy remains
unvalidated. Quality proxies do not establish detection gains.

In [ ]:
policy = validate_policy(read_json(FROZEN_POLICY), protocol)
freeze_json(RUN_ROOT / "policy.json", policy)
print("Frozen policy before pilot inspection:", policy["policy_id"])
display(policy["recipes"])
display(
    pd.DataFrame(policy["route_support"]).T[
        ["eligible_count", "candidate_count", "minimum_support", "status"]
    ]
)
display(policy["noise_diagnostics"])
display(pd.DataFrame(policy["candidates"]))
calibration = pd.DataFrame(policy["measurements"])
calibration[list(METRICS[:4])].hist(bins=24, figsize=(10, 5))
plt.suptitle("Calibration sample · quality proxies only")
plt.tight_layout()
plt.savefig(REPORT_ROOT / "calibration_quality.png")
plt.show()
print("\n".join(policy["limits"]))

## Budget storage, convert once, export exactly Raw and Adaptive

Estimate from an evenly spaced selected-training sample using the frozen recipe.
Use the p90 image-size estimate plus 25% headroom and 1 GiB for labels/conversion.
This is an estimate, not a worst-case bound. Only the selected subset is exported;
S20/S40 do not materialize automatically. Sources stay unchanged. Labels are converted
once through the official stack with 0.01-pixel source-box roundtrip assertions, then
copied byte-for-byte. Pair identity pins the protocol, subset, policy and exporter.
Neither branch contains development-validation or final-test images.

In [ ]:
records, _, sources = scan_bdd(DATA_ROOT, strict=False)
for split, source in sources.items():
    assert source["sha256"] == protocol["annotation_sources"][split]["sha256"]
partitions = training_records(protocol, subsets, EXPORT_SUBSET, records, DATA_ROOT)
selected = partitions["train"]
sample = [selected[i] for i in np.linspace(0, len(selected) - 1, min(16, len(selected)), dtype=int)]
budget = estimate_disk(sample, len(selected), ["raw_reencoded", "adaptive"], policy["recipes"])
budget.to_csv(REPORT_ROOT / "storage_estimate.csv", index=False)
required = float(budget.p90_budget_GiB.sum() * 1.25 + 1)
free = shutil.disk_usage(RUN_ROOT).free / 2**30
display(budget)
print(f"Estimated headroom requirement {required:.2f} GiB; free {free:.2f} GiB")
assert free >= required, "Insufficient estimated storage; do not start the export"
converted = RUN_ROOT / "conversion" / EXPORT_SUBSET
with capture_output() as conversion_log:
    conversion = convert_labels(partitions, converted, convention="continuous")
(RUN_ROOT / f"{EXPORT_SUBSET}_conversion.log").write_text(
    conversion_log.stdout + conversion_log.stderr
)
pair_root = RUN_ROOT / "datasets" / EXPORT_SUBSET
tables = export_pair(partitions, converted, pair_root, policy, protocol, subsets, EXPORT_SUBSET)
assert verify_alignment([pair_root / b for b in tables]) == len(selected)
for branch in tables:
    assert not list((pair_root / branch / "images/test").iterdir())
    assert not list((pair_root / branch / "images/tune").iterdir())
print("Deep-verified paired images per branch:", len(selected))

## Paired JPEG quality, routing and offline cost

Before/after below compares the **decoded exported Raw JPEG95** and **Adaptive JPEG95**.
Signed deltas expose adverse changes; higher brightness, contrast or sharpness alone
is not success. Policy decisions use image pixels only. Environmental labels are joined
**after freeze** for descriptive subgroup analysis. Preprocessing times include metric
extraction and filtering on CPU, exclude decode/encode/write, and retain their original
measurement on export reuse. Report full preparation separately. These are offline
costs on this machine, not detector or end-to-end deployment latency.

In [ ]:
paired = paired_quality(pair_root)
paired.to_csv(pair_root / "paired_quality.csv", index=False)
meta = tables["adaptive"].copy()
meta["filters"] = meta.decisions.map(len)
meta["decision"] = meta.decisions.map(lambda d: d[0]["name"] if d else "skip")
assert meta.filters.max() <= 1
attributes = pd.DataFrame([{"id": r["id"], **r["attributes"]} for r in pool])
analysis = paired.merge(
    meta[["id", "decision", "filters", "preprocess_ms", "preparation_ms"]],
    on="id",
    validate="one_to_one",
).merge(attributes, on="id", validate="one_to_one")
analysis["brightness_stratum"] = pd.cut(
    analysis.raw_brightness,
    [0, 0.2, 0.4, 1],
    labels=["dark", "middle", "bright"],
    include_lowest=True,
)
analysis["noise_stratum"] = np.where(analysis.raw_noise > 0.035, "noisy proxy", "low noise proxy")
analysis.to_csv(pair_root / "analysis.csv", index=False)
display(analysis.decision.value_counts().rename("images"))
print("Skip rate:", float((analysis.decision == "skip").mean()))
print("Filter counts:", analysis.filters.value_counts().to_dict())
latency = pd.DataFrame(
    [
        {
            "branch": b,
            "stage": stage,
            "mean_ms": t[stage].mean(),
            "p50_ms": t[stage].quantile(0.5),
            "p95_ms": t[stage].quantile(0.95),
        }
        for b, t in tables.items()
        for stage in ["preprocess_ms", "preparation_ms"]
    ]
)
display(latency)
latency.to_csv(REPORT_ROOT / "offline_latency.csv", index=False)
fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for j, metric in enumerate(METRICS[:4]):
    analysis[[f"raw_{metric}", f"adaptive_{metric}"]].plot.hist(
        ax=axes[0, j], bins=32, alpha=0.5, title=metric
    )
    analysis[f"delta_{metric}"].plot.hist(ax=axes[1, j], bins=32, title=f"Paired delta {metric}")
    axes[1, j].axvline(0, color="black", linewidth=1)
fig.suptitle("Image-quality proxies · no detection metrics")
fig.tight_layout()
fig.savefig(REPORT_ROOT / "paired_quality.png")
plt.show()
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, axis in zip(axes, ["weather", "timeofday", "scene"], strict=True):
    pd.crosstab(analysis[axis], analysis.decision, normalize="index").plot.bar(
        stacked=True, ax=ax, title=f"Decision fraction by {axis}", rot=65
    )
fig.tight_layout()
fig.savefig(REPORT_ROOT / "routing_environment.png")
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, axis in zip(axes, ["brightness_stratum", "noise_stratum"], strict=True):
    pd.crosstab(analysis[axis], analysis.decision, normalize="index").plot.bar(
        stacked=True, ax=ax, title=axis.replace("_", " "), rot=20
    )
fig.tight_layout()
fig.savefig(REPORT_ROOT / "routing_quality.png")
plt.show()
display(analysis.groupby("decision").proxy_loss_delta.agg(["count", "mean", "min", "max"]))

In [ ]:
changed = analysis[analysis.decision != "skip"].sort_values(["proxy_loss_delta", "id"])
examples = []
if len(changed):
    examples += [
        ("Best proxy change", changed.iloc[0]["id"]),
        ("Worst proxy change / adverse candidate", changed.iloc[-1]["id"]),
    ]
    clipping = changed.sort_values(["delta_white_fraction", "id"])
    examples += [("Largest white-saturation change / failure candidate", clipping.iloc[-1]["id"])]
skipped = analysis[analysis.decision == "skip"].sort_values("id")
if len(skipped):
    examples.append(("Skipped control", skipped.iloc[0]["id"]))
fig, axes = plt.subplots(len(examples), 2, figsize=(12, 3 * len(examples)), squeeze=False)
for i, (label, sample_id) in enumerate(examples):
    for j, branch in enumerate(["raw", "adaptive"]):
        row = tables[branch].set_index("id").loc[sample_id]
        axes[i, j].imshow(cv2.cvtColor(read_image(row.image), cv2.COLOR_BGR2RGB))
        axes[i, j].set_title(f"{label} · {branch} · {sample_id}", fontsize=8)
        axes[i, j].axis("off")
fig.suptitle("Representative visual checks · proxy ranking, not detector success/failure")
fig.tight_layout()
# Source-image examples stay gitignored; the notebook retains inline inspection evidence.
fig.savefig(pair_root / "examples.png")
plt.show()

## Prepare matched development inputs, without tuning

Decode original development sources for Adaptive; use the same converted labels,
dimensions and JPEG95 as Raw. The frozen recipe is applied without calibration.
No final-test images are exported.

In [ ]:
development = export_development_pair(
    protocol, policy, RAW_DEVELOPMENT, DEV_CONVERSION, RUN_ROOT / "development"
)
print("Verified development inputs:", development)

## Save reproducible compact evidence

The complete immutable subset manifest and calibrated recipe remain under the run root.
Commit compressed training-pool/subset membership, the frozen small recipe, hashes,
support/quality/cost summaries and figures. Images and per-image large tables stay under
runs; final test stays untouched. S20 is ready for a later paired scaling decision.
Do not retune this recipe after inspecting downstream development results; a new
research decision requires a new version and disclosure.

In [ ]:
subset_ids = {name: set(s["ids"]) for name, s in subsets["subsets"].items()}
membership = pd.DataFrame(
    [
        {
            "id": r["id"],
            "group": r["group"],
            **{name: r["id"] in subset_ids[name] for name, s in subsets["subsets"].items()},
        }
        for r in sorted(pool, key=lambda r: r["id"])
    ]
)
member_path = REPORT_ROOT / "subset_membership.csv.gz"
compressed = gzip.compress(membership.to_csv(index=False).encode(), mtime=0)
if member_path.exists() and member_path.read_bytes() != compressed:
    raise ValueError("Committed subset membership changed; choose a new version/root")
member_path.write_bytes(compressed)
write_json(REPORT_ROOT / "policy.json", policy)
summary = {
    "protocol_id": protocol["manifest_id"],
    "subsets_id": subsets["manifest_id"],
    "policy_id": policy["policy_id"],
    "policy_version": policy["version"],
    "route_support": policy["route_support"],
    "noise_diagnostics": policy["noise_diagnostics"],
    "membership_sha256": file_hash(member_path),
    "subsets": {
        name: {k: v for k, v in s.items() if k not in {"ids", "groups"}}
        for name, s in subsets["subsets"].items()
    },
    "materialized_subset": EXPORT_SUBSET,
    "images_per_branch": len(selected),
    "pair": read_json(pair_root / "pair.json"),
    "exports": {
        b: {
            "identity": read_json(pair_root / b / "export.json"),
            "verified": read_json(pair_root / b / "verified.json"),
        }
        for b in tables
    },
    "skip_rate": float((analysis.decision == "skip").mean()),
    "decisions": {k: int(v) for k, v in analysis.decision.value_counts().items()},
    "proxy_improved": int((analysis.proxy_loss_delta < -1e-8).sum()),
    "proxy_worsened": int((analysis.proxy_loss_delta > 1e-8).sum()),
    "mean_proxy_loss_delta": float(analysis.proxy_loss_delta.mean()),
    "examples": examples,
    "offline_latency": latency.to_dict("records"),
    "actual_image_GiB": {
        b: sum(Path(p).stat().st_size for p in t.image) / 2**30 for b, t in tables.items()
    },
    "environment": {
        "python": platform.python_version(),
        "opencv": cv2.__version__,
        "numpy": np.__version__,
        "cpu_count": os.cpu_count(),
        "opencv_threads": cv2.getNumThreads(),
        "machine": platform.machine(),
    },
    "final_test_pixels_accessed": False,
    "development_pixels_accessed": True,
    "detector_run": False,
    "training_run": False,
    "support_audit_sha256": file_hash(REPORT_ROOT / "subset_support.csv"),
    "paired_quality_sha256": file_hash(pair_root / "paired_quality.csv"),
    "limitations": policy["limits"] + subsets["limitations"],
}
write_json(REPORT_ROOT / "summary.json", summary)
print("PASS: nested immutable subsets; calibration-only frozen recipe; paired JPEG95 S10 exports")
display(
    pd.Series(
        {
            "S10": len(subsets["subsets"]["S10"]["ids"]),
            "S20": len(subsets["subsets"]["S20"]["ids"]),
            "Raw GiB": summary["actual_image_GiB"]["raw"],
            "Adaptive GiB": summary["actual_image_GiB"]["adaptive"],
        }
    )
)

## Synthetic screening evidence and policy decision

The calibration-only screen and its corrected fixed-class analysis are archived,
including executed plots and inference identities. No new screen is run here.
The selected-pilot CLAHE finding did not validate a new router: retain Gamma-or-Skip.
[Research snapshots](archive/README.md) contain full ablations, estimator diagnostics
and the metadata-only final robustness specification.

In [ ]:
from IPython.display import Image

decision = read_json(ROOT / "docs/issue11-v2/v3_decision.json")
assert decision["status"] == "v3_not_deployed"
assert decision["retained_policy_id"] == policy["policy_id"]
display(pd.read_csv(ROOT / "docs/issue11-v2/experimental_matrix.csv").head(12))
display(Image(filename=str(ROOT / "docs/issue11-v2/supported_recovery.png"), width=900))
print(decision["reason"])